# Hybrid retrieval trên Zalo Legal

Corpus: **61 425 điều luật** Việt Nam, 3 298 câu hỏi có nhãn điều đúng (Zalo AI Legal Text Retrieval). Bạn dựng
hai nhánh retrieval trên **toàn bộ corpus**, BM25 (theo từ) và dense `bge-m3` (theo nghĩa), rồi trộn bằng
Reciprocal Rank Fusion và đo trên tập đánh giá ghim: **300 câu hỏi đầu** của `qrels/test.jsonl`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama đang chạy với `bge-m3` (`ollama pull bge-m3`). Lần đầu embed cả corpus mất khoảng 45 phút trên
  laptop GPU 4GB; vector được cache ở `data/raw/zalo-legal/cache/` nên lần sau chỉ mất vài giây.
- Chấm: `uv run pytest tracks/ai-engineer/modules/13-rag/exercises/intermediate/01-hybrid-retrieval -q`

In [ ]:
# explore: tìm gốc repo, tải Zalo Legal nếu chưa có (khoảng 110 MB, chỉ lần đầu)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

ZALO = ensure("zalo-legal")
CACHE = ZALO / "cache"
CACHE.mkdir(exist_ok=True)
print(ZALO, sorted(p.name for p in ZALO.iterdir()))

In [ ]:
import json
import math
import re
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np

## 1. Đọc corpus và tập đánh giá

Dữ liệu ở dạng BEIR, mỗi dòng một JSON:

| Tệp | Một dòng |
| --- | --- |
| `corpus.jsonl` | `{"_id": "100/2019/nđ-cp+5", "title": "Điều 5. ...", "text": "1. ..."}` (`_id` = số hiệu văn bản + `+` + số điều) |
| `queries.jsonl` | `{"_id": "268afbc1...", "text": "Mức phạt khi quay đầu xe ô tô trên đường cao tốc"}` |
| `qrels/test.jsonl`, `qrels/train.jsonl` | `{"query-id": "268afbc1...", "corpus-id": "100/2019/nđ-cp+5", "score": 1}` |

- `load_corpus(root) -> list[Article]`: mọi điều theo thứ tự tệp, `uid` là `_id`.
- `load_questions(root, n, split="test") -> list[Question]`: lấy `n` **query-id phân biệt đầu tiên** theo thứ tự
  xuất hiện trong `qrels/<split>.jsonl`; `relevant` là tập mọi `corpus-id` của câu đó trong tệp (một câu có thể có
  nhiều điều đúng, các dòng của nó không nhất thiết liền nhau); `text` lấy từ `queries.jsonl`. Có ít hơn `n` câu thì
  trả hết.

Tập đánh giá ghim của cả ba bài Intermediate là `load_questions(root, 300)`: split `test`, để dành `train` cho
milestone fine-tune embedding của capstone.

In [ ]:
@dataclass
class Article:
    uid: str      # _id trong corpus.jsonl, vd. "100/2019/nđ-cp+5"
    title: str
    text: str


@dataclass
class Question:
    qid: str
    text: str
    relevant: set[str]   # uid của các điều đúng


def load_corpus(root: Path) -> list[Article]:
    """Mọi dòng của root/corpus.jsonl thành Article, giữ thứ tự tệp."""
    # TODO
    raise NotImplementedError


def load_questions(root: Path, n: int, split: str = "test") -> list[Question]:
    """n query-id phân biệt đầu tiên theo thứ tự root/qrels/<split>.jsonl, kèm text và tập điều đúng."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc corpus và 300 câu ghim
t0 = time.perf_counter()
articles = load_corpus(ZALO)
questions = load_questions(ZALO, 300)
print(f"{len(articles)} điều, {len(questions)} câu hỏi ({time.perf_counter() - t0:.1f} s)")
print("câu đầu:", questions[0])
print("số điều đúng mỗi câu:", Counter(len(q.relevant) for q in questions))
laws = {uid.rsplit("+", 1)[0] for q in questions for uid in q.relevant}
print("số văn bản chứa đáp án:", len(laws))
lengths = np.array([len(a.title) + 1 + len(a.text) for a in articles])
print("độ dài điều (ký tự): trung vị", int(np.median(lengths)), "| p99", int(np.percentile(lengths, 99)),
      "| dài hơn 16 000 (bge-m3 qua Ollama cắt ở 4 096 token):", int((lengths > 16000).sum()))

## 2. Nhánh BM25 và thước đo (dán từ bài Basic)

Notebook này tự chứa: **dán** lời giải của bạn ở bài Basic 02 (`tokenize_vi`, `BM25`) và Basic 03 (`recall_at_k`,
`mrr_at_k`) vào cell dưới, giữ nguyên chữ ký. Đừng import chéo bài. Một test nhỏ kiểm lại chúng để bảng ở cuối bài
đáng tin.

In [ ]:
def tokenize_vi(text: str) -> list[str]:
    """NFC, chữ thường, bỏ dấu câu, tách theo khoảng trắng (Basic 02)."""
    # TODO: dán từ bài Basic 02
    raise NotImplementedError


class BM25:
    """BM25 với IDF dạng Lucene (Basic 02)."""

    def __init__(self, docs: list[list[str]], k1: float = 1.5, b: float = 0.75):
        # TODO: dán từ bài Basic 02
        raise NotImplementedError

    def scores(self, query: list[str]) -> np.ndarray:
        # TODO: dán từ bài Basic 02
        raise NotImplementedError

    def top_k(self, query: list[str], k: int) -> list[int]:
        # TODO: dán từ bài Basic 02
        raise NotImplementedError


def recall_at_k(retrieved: list[list[str]], relevant: list[set[str]], k: int) -> float:
    # TODO: dán từ bài Basic 03
    raise NotImplementedError


def mrr_at_k(retrieved: list[list[str]], relevant: list[set[str]], k: int) -> float:
    # TODO: dán từ bài Basic 03
    raise NotImplementedError

In [ ]:
# explore: index BM25 cả 61 425 điều (title + text), xếp hạng top-100 cho 300 câu
uids = [a.uid for a in articles]
relevant = [q.relevant for q in questions]

t0 = time.perf_counter()
bm25 = BM25([tokenize_vi(a.title + " " + a.text) for a in articles])
print(f"dựng index BM25: {time.perf_counter() - t0:.1f} s, {len(bm25.idf)} âm tiết khác nhau")

t0 = time.perf_counter()
bm25_rank = [bm25.top_k(tokenize_vi(q.text), 100) for q in questions]
print(f"300 truy vấn BM25: {1000 * (time.perf_counter() - t0) / len(questions):.0f} ms/câu")


def report(name: str, ranked_idx: list[list[int]]) -> dict:
    ranked = [[uids[i] for i in r] for r in ranked_idx]
    row = {"R@1": recall_at_k(ranked, relevant, 1), "R@5": recall_at_k(ranked, relevant, 5),
           "R@10": recall_at_k(ranked, relevant, 10), "R@100": recall_at_k(ranked, relevant, 100),
           "MRR@10": mrr_at_k(ranked, relevant, 10)}
    print(f"{name:28s}" + "  ".join(f"{k} {v:.3f}" for k, v in row.items()))
    return row


table = {"BM25 âm tiết": report("BM25 âm tiết", bm25_rank)}

## 3. Nhánh dense: embed corpus và tìm theo cosine

- `embed_corpus(texts, embed_fn, batch_size=32) -> np.ndarray`: gọi `embed_fn` lần lượt trên từng lô
  `batch_size` văn bản liên tiếp (lô cuối có thể ngắn hơn), ghép kết quả thành ma trận `float32` shape
  `(len(texts), dim)` theo đúng thứ tự, rồi **chuẩn hoá L2** từng dòng. Dòng toàn 0 (embed lỗi) giữ nguyên 0,
  không được thành `NaN`.
- `dense_search(q_vec, matrix, k) -> list[int]`: chỉ số `k` dòng của `matrix` có tích vô hướng với `q_vec` lớn
  nhất, giảm dần; bằng điểm thì chỉ số nhỏ đứng trước. Vì các dòng đã chuẩn hoá, xếp theo tích vô hướng chính là
  xếp theo cosine (độ dài của `q_vec` không đổi thứ hạng).

`embed_fn` là bất kỳ hàm `list[str] -> np.ndarray` nào: `llmkit.embed` khi chạy thật, `FakeLLM().embed` trong
test. Tách nó ra tham số để test không cần Ollama.

In [ ]:
def embed_corpus(texts: list[str], embed_fn: Callable[[list[str]], np.ndarray],
                 batch_size: int = 32) -> np.ndarray:
    """Embed theo lô batch_size, ghép theo thứ tự, chuẩn hoá L2 từng dòng (dòng 0 giữ 0). float32 (N, dim)."""
    # TODO
    raise NotImplementedError


def dense_search(q_vec: np.ndarray, matrix: np.ndarray, k: int) -> list[int]:
    """k chỉ số dòng có tích vô hướng với q_vec lớn nhất, giảm dần; bằng điểm thì chỉ số nhỏ trước."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: embed cả corpus bằng bge-m3 (cache ra đĩa), rồi 300 câu hỏi
import llmkit


def safe_embed(texts: list[str]) -> np.ndarray:
    """llmkit.embed, nhưng một văn bản lỗi (bge-m3 đôi khi trả NaN, Ollama báo HTTP 500) không làm hỏng cả lần chạy 45 phút.

    llmkit đã tự gửi lại từng văn bản khi cả lô lỗi; văn bản vẫn lỗi thì thử sửa nhẹ, cuối cùng cho vector 0.
    """
    try:
        return llmkit.embed(texts)
    except llmkit.LLMError:
        rows = []
        for text in texts:
            for variant in (text, text.strip() + " .", text[: len(text) // 2]):
                try:
                    rows.append(llmkit.embed([variant])[0])
                    break
                except llmkit.LLMError:
                    continue
            else:
                print("không embed được:", text[:80])
                rows.append(np.zeros(1024, np.float32))
        return np.stack(rows)


matrix_path, ids_path = CACHE / "bge-m3.npy", CACHE / "bge-m3.ids.json"
if matrix_path.exists() and ids_path.exists() and json.loads(ids_path.read_text(encoding="utf-8")) == uids:
    matrix = np.load(matrix_path)
    print("đọc cache", matrix_path, matrix.shape)
else:
    print("embed", len(articles), "điều, khoảng 45 phút trên GPU 4GB ...")
    t0 = time.perf_counter()
    matrix = embed_corpus([a.title + "\n" + a.text for a in articles], safe_embed, batch_size=32)
    elapsed = time.perf_counter() - t0
    print(f"embed corpus: {elapsed / 60:.1f} phút ({len(articles) / elapsed:.1f} điều/giây)")
    np.save(matrix_path, matrix)
    ids_path.write_text(json.dumps(uids, ensure_ascii=False), encoding="utf-8")
print("số dòng toàn 0 (embed lỗi):", int((np.abs(matrix).sum(axis=1) == 0).sum()))

t0 = time.perf_counter()
q_matrix = embed_corpus([q.text for q in questions], safe_embed)
print(f"embed 300 câu hỏi: {time.perf_counter() - t0:.1f} s")

t0 = time.perf_counter()
dense_rank = [dense_search(v, matrix, 100) for v in q_matrix]
print(f"dense search: {1000 * (time.perf_counter() - t0) / len(questions):.0f} ms/câu trên {len(matrix)} vector")
table["Dense bge-m3"] = report("Dense bge-m3", dense_rank)

top1 = np.array([float(matrix[r[0]] @ v) for r, v in zip(dense_rank, q_matrix)])
print(f"cosine cao nhất mỗi câu: trung vị {np.median(top1):.3f}, thấp nhất {top1.min():.3f}")

## 4. Reciprocal Rank Fusion

$$
\mathrm{RRF}(d) = \sum_{r \in R} \frac{1}{k + \mathrm{rank}_r(d)}
$$

`rrf_fuse(rankings, k=60) -> list[int]`: `rankings` là các danh sách chỉ số tài liệu, mỗi danh sách đã xếp giảm dần
và không trùng; hạng bắt đầu từ 1; tài liệu vắng mặt ở một danh sách không nhận gì từ danh sách đó. Trả **mọi**
tài liệu xuất hiện ở ít nhất một danh sách, xếp theo điểm RRF giảm dần; bằng điểm thì chỉ số nhỏ đứng trước.
Kiểm bằng bảng tính tay ở mục 3 lý thuyết (BM25 xếp A, B, C; dense xếp C, A, D).

In [ ]:
def rrf_fuse(rankings: list[list[int]], k: int = 60) -> list[int]:
    """Trộn các danh sách xếp hạng bằng RRF; điểm giảm dần, bằng điểm thì chỉ số nhỏ trước."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: kiểm bảng tính tay (A=0, B=1, C=2, D=3), rồi hybrid trên 300 câu
print("RRF bảng lý thuyết:", rrf_fuse([[0, 1, 2], [2, 0, 3]]), "(mong đợi [0, 2, 1, 3] = A, C, B, D)")

hybrid_rank = [rrf_fuse([b, d])[:100] for b, d in zip(bm25_rank, dense_rank)]
table["Hybrid RRF (BM25 + dense)"] = report("Hybrid RRF (BM25 + dense)", hybrid_rank)

# lưu top-100 hybrid cho bài 03 (rerank) dùng lại, không phải dựng lại hai index
_ = (CACHE / "hybrid-top100.json").write_text(
    json.dumps({q.qid: [uids[i] for i in r] for q, r in zip(questions, hybrid_rank)}, ensure_ascii=False),
    encoding="utf-8")

In [ ]:
# explore: bảng tổng hợp và nhánh nào cứu câu nào
import pandas as pd

display(pd.DataFrame(table).T.round(3))


def hit(ranked_idx: list[int], q: Question, k: int) -> bool:
    return any(uids[i] in q.relevant for i in ranked_idx[:k])


both = sum(hit(b, q, 10) and hit(d, q, 10) for b, d, q in zip(bm25_rank, dense_rank, questions))
only_bm25 = sum(hit(b, q, 10) and not hit(d, q, 10) for b, d, q in zip(bm25_rank, dense_rank, questions))
only_dense = sum(hit(d, q, 10) and not hit(b, q, 10) for b, d, q in zip(bm25_rank, dense_rank, questions))
neither = len(questions) - both - only_bm25 - only_dense
print(f"top-10 có điều đúng: cả hai {both}, chỉ BM25 {only_bm25}, chỉ dense {only_dense}, không nhánh nào {neither}")
print("hybrid top-10 có điều đúng:", sum(hit(h, q, 10) for h, q in zip(hybrid_rank, questions)), "/", len(questions))

for b, d, q in zip(bm25_rank, dense_rank, questions):
    if hit(b, q, 1) and not hit(d, q, 10):
        print("\nBM25 đúng hạng 1, dense trượt top-10:", q.text, "->", sorted(q.relevant))
        break
for b, d, q in zip(bm25_rank, dense_rank, questions):
    if hit(d, q, 1) and not hit(b, q, 10):
        print("Dense đúng hạng 1, BM25 trượt top-10:", q.text, "->", sorted(q.relevant))
        break

## Đọc kết quả

So với bảng trên **lát cắt** ở mục 3 lý thuyết (100 câu, 3 541 điều cùng văn bản với đáp án), corpus ở đây lớn gấp
17 lần và có cả điều của văn bản khác làm nhiễu. Tự trả lời trước khi làm trắc nghiệm:

- Nhánh nào tụt nhiều hơn khi corpus lớn lên? Vì sao một âm tiết hiếm như "đê" vẫn giúp BM25, còn câu hỏi chung chung
  ("thủ tục ...", "hồ sơ ...") thì không?
- Hybrid có thắng **cả hai** nhánh ở mọi cột không? Cột nào thì không, và vì sao RRF có thể kéo một câu đang đúng ở
  hạng 1 xuống?
- R@100 của hybrid là trần cho reranker ở bài 03: rerank top-50 không cứu được câu nào điều đúng nằm ngoài top-50.

Sau đó làm phần trắc nghiệm của bài trong app.